# Target acquisition

Take overview images, let the analysis find objects, then image the brightest few at high magnification.

It runs on the mock microscope. Run `python install.py --branch release-candidate` in this repository first.

## 1. Connect

Discover the microscopes that are there and select the one for Leica.

In [ ]:
# The ZMART Controller: one way to talk to every microscope
from zmart_controller import mic

# List the microscopes configured on this computer
mic.get_instruments()


<br>

Connect to it. Here we connect to the mock microscope.

In [ ]:
# Connect to the mock microscope; its images are saved in the "images" folder
mic.connect("mock", {"output_root": "images"})


<br>

## 2. Positions

Set up a list.

In [ ]:
# An empty list; each position you note down is added to it
positions = []


<br>

Move the stage to position 1 and note down the location.

In [ ]:
# Ask the microscope where the stage is now
xyz = mic.get_xyz()["output"]

# Add this position (x, y, z in micrometres) to the list
positions.append([xyz["x"]["position"], xyz["y"]["position"], xyz["z"]["position"]])


<br>

Move the stage to position 2 and note down the location.

In [ ]:
# Ask the microscope where the stage is now
xyz = mic.get_xyz()["output"]

# Add this position (x, y, z in micrometres) to the list
positions.append([xyz["x"]["position"], xyz["y"]["position"], xyz["z"]["position"]])


<br>

## 3. States

Set up a list.

In [ ]:
# An empty list; each state you note down is added to it
states = []


<br>

Set the microscope to the overview settings and note down the state.

In [ ]:
# Note down the overview settings (objective, laser, exposure, ...)
states.append(mic.get_state()["output"])


<br>

Set the microscope to the target settings and note down the state.

In [ ]:
# Note down the target settings
states.append(mic.get_state()["output"])


<br>

## 4. Analysis

Set up the analysis that finds the objects.

In [ ]:
# The ZMART analysis engine
from zmart_analysis import Engine

# The pipeline: finds nuclei with a watershed and measures each one
pipeline = "../../work/ZMART-analysis/workflows/object_analysis/pipelines/object_analysis_watershed.yaml"

# Start the analysis
analysis = Engine()

# Name the pipeline "detect_targets"; its results go to the "analysis_output" folder
analysis.register("detect_targets", pipeline, "analysis_output")


<br>

Set up a list for the analysis jobs.

In [ ]:
# An empty list; each overview image sent to the analysis adds its job to it
jobs = []


<br>

## 5. The loop

Image the overview positions, find the objects, then image the brightest ones.

In [ ]:
# To wait a moment while the analysis runs
import time

# How many objects to image per overview
N_TARGETS = 5

# 1. Overview: take an image at each position and send it to the analysis.
# Go through the positions one by one
for i, (x, y, z) in enumerate(positions):
    # Move the stage there
    mic.set_xyz(x, y, z)

    # Switch to the overview settings
    mic.set_state(states[0])

    # Take the image
    image = mic.acquire(position_label=f"overview{i}")["output"]

    # The size of one pixel in the overview settings, in micrometres
    pixel_um = states[0]["observed"]["pixel_size"]["x"]

    # Send the image to the analysis; it works in the background
    jobs.append(analysis.submit("detect_targets", {
        "image_path": image["files"][0],                   # the image file
        "tile_id": [f"overview{i}"],                       # a name for this image
        "tile_stage_xy_um": [x, y],                        # where the stage was
        "source_pixel_size_um": [pixel_um, pixel_um],      # pixel size in x and y
        "source_image_size_px": [64, 64],                  # image size in pixels
        "image_to_stage": [[1, 0], [0, 1]],                # image orientation on the stage
    }))

# 2. Wait for the analysis, then choose the brightest objects of each overview.
# The objects to image
targets = []

# Go through the analysis jobs one by one
for i, job in enumerate(jobs):
    # Ask the analysis how this job is doing
    entry = analysis.status("detect_targets")[job]

    # Until it is done, wait a moment and ask again
    while entry["state"] not in ("done", "failed"):
        time.sleep(0.2)
        entry = analysis.status("detect_targets")[job]

    # Stop here if the analysis failed, and show why
    assert entry["state"] == "done", entry.get("error")

    # The table of objects: one row per object, with its brightness and stage position
    table = entry["output"]["object_analysis"]["objects"]["properties"]

    # The objects, brightest first
    brightest = sorted(range(len(table["label"])), key=lambda k: -table["intensity_mean"][k])

    # Keep the brightest few, with a name and their stage position
    for k in brightest[:N_TARGETS]:
        targets.append((f"overview{i}_object{table['label'][k]}",
                        table["stage_x_um"][k], table["stage_y_um"][k], positions[i][2]))

# 3. Targets: switch to the close-up state and visit each chosen object.
# Go through the targets one by one
for label, x, y, z in targets:
    # Move the stage to the object
    mic.set_xyz(x, y, z)

    # Switch to the target settings
    mic.set_state(states[1])

    # Take the image
    mic.acquire(position_label=label)


<br>

Close the connections.

In [ ]:
# Stop the analysis
analysis.shutdown()

# Close the connection to the microscope
mic.disconnect()
